# Ti MACE potential on the full v3 data (GPU): training, held-out test, melting point
Part of *Thermal or non-thermal?* (Omeje & Itina). Runs on a free Colab T4.

**Before you start:** Runtime → Change runtime type → **T4 GPU**. Put `mace_v3.zip` (made on your computer with `scripts/mace_prepare_v3.py`) in the Google Drive folder **`ti_mace_v3`**.

Everything that must survive a disconnect (checkpoints, model, logs, results) is written to that Drive folder. If the session drops, reconnect and **run all cells again**: training resumes from the last checkpoint, and finished melting temperatures are skipped.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
WORK = '/content/drive/MyDrive/ti_mace_v3'
os.makedirs(WORK, exist_ok=True)
assert os.path.exists(f'{WORK}/mace_v3.zip'), 'Upload mace_v3.zip to the Drive folder ti_mace_v3 first'
!mkdir -p /content/data && cd /content/data && unzip -o -q {WORK}/mace_v3.zip && ls -la /content/data

In [ ]:
!pip install -q mace-torch==0.3.17 && python -c "import mace, torch; print('mace', mace.__version__, 'torch', torch.__version__, 'cuda', torch.cuda.is_available())"

In [ ]:
%%writefile /content/data/mace_test_eval.py
"""Held-out test errors of a MACE model on test_Init.xyz and test_Bulk.xyz. usage: mace_test_eval.py model [device]"""
import sys, numpy as np
from ase.io import read
from mace.calculators import MACECalculator
calc = MACECalculator(model_paths=sys.argv[1], device=sys.argv[2] if len(sys.argv) > 2 else 'cuda', default_dtype='float32')
for g in ('Init', 'Bulk'):
    de, df = [], []
    for a in read(f'test_{g}.xyz', ':'):
        e0, f0 = a.info['REF_energy'], a.arrays['REF_forces']; a.calc = calc
        de.append((a.get_potential_energy() - e0) / len(a)); df.append((a.get_forces() - f0).ravel())
    print(f'{g:5s} held-out: energy RMSE {1000 * np.sqrt(np.mean(np.square(de))):6.1f} meV/atom, '
          f'force RMSE {np.sqrt(np.mean(np.square(np.concatenate(df)))):.3f} eV/A  ({len(de)} configs)')


In [ ]:
%%writefile /content/data/mace_melt.py
"""Melting point of Ti from solid-liquid coexistence with a MACE model (ASE, GPU).
bcc slab (6x6x14 cells, 1008 atoms): equilibrate at T, melt the upper half at 3000 K with the lower half fixed,
then run the two-phase box at T and zero pressure (Berendsen) for t_coex ps. If the potential energy falls, the
liquid crystallises (T < Tm); if it rises, the solid melts (T > Tm). usage: mace_melt.py model T [t_coex_ps] [device]"""
import sys, numpy as np
from ase.build import bulk
from ase import units
from ase.constraints import FixAtoms
from ase.md.langevin import Langevin
from ase.md.nptberendsen import NPTBerendsen
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution
from mace.calculators import MACECalculator
model, T = sys.argv[1], float(sys.argv[2]); tco = float(sys.argv[3]) if len(sys.argv) > 3 else 30.0
dev = sys.argv[4] if len(sys.argv) > 4 else 'cuda'; REP = (6, 6, 14) if dev == 'cuda' else (2, 2, 4)
dt = 2.0 * units.fs; kappa = 1 / (110 * units.GPa)                  # compressibility for the barostat
at = bulk('Ti', 'bcc', a=3.28, cubic=True).repeat(REP)
at.calc = MACECalculator(model_paths=model, device=dev, default_dtype='float32')
MaxwellBoltzmannDistribution(at, temperature_K=T, rng=np.random.default_rng(1))
npt = lambda: NPTBerendsen(at, dt, temperature_K=T, taut=100 * units.fs, pressure_au=0.0, taup=1000 * units.fs,
                           compressibility_au=kappa)
npt().run(2500)                                                      # 5 ps solid at T, zero pressure
z = at.positions[:, 2]; top = z > 0.5 * at.cell[2, 2]
at.set_constraint(FixAtoms(mask=~top))
Langevin(at, dt, temperature_K=3000, friction=0.02, rng=np.random.default_rng(2)).run(2000)   # 4 ps: melt the top
at.set_constraint(); MaxwellBoltzmannDistribution(at, temperature_K=T, rng=np.random.default_rng(3))
md = npt(); out = []
def log(): out.append((len(out) * 0.1, at.get_potential_energy() / len(at), at.get_temperature(), at.get_volume() / len(at)))
md.attach(log, interval=50); md.run(int(tco * 500))
d = np.array(out); np.savetxt(f'melt_T{T:.0f}.dat', d, header='t_ps pe_eV_per_atom T_K V_A3_per_atom')
n = len(d) // 3; slope = np.polyfit(d[n:, 0], d[n:, 1], 1)[0] * 1000
print(f'T = {T:.0f} K: dE/dt over the last 2/3 = {slope:+.3f} meV/atom/ps -> '
      f'{"solid grows (T < Tm)" if slope < 0 else "solid melts (T > Tm)"};  V = {d[-1, 3]:.2f} A3/atom')


## 1. Training (about 3–6 h on a T4; resumable)
Same settings as the small CPU model, but larger: 64 channels, L_max = 1, correlation order 3, two interactions, 5 Å cutoff; 150 epochs, forces weighted first and energies raised in the second stage (epoch 110).

In [ ]:
%cd /content/data
!mace_run_train --name=ti_mace_v3 --train_file=train.xyz --valid_file=valid.xyz \
  --energy_key=REF_energy --forces_key=REF_forces --E0s=average \
  --model=MACE --r_max=5.0 --num_interactions=2 --num_channels=64 --max_L=1 --correlation=3 \
  --batch_size=4 --valid_batch_size=8 --max_num_epochs=150 --lr=0.01 --amsgrad \
  --ema --ema_decay=0.99 --swa --start_swa=110 --eval_interval=2 \
  --device=cuda --default_dtype=float32 --seed=1 --restart_latest \
  --results_dir={WORK}/res --checkpoints_dir={WORK}/ck --model_dir={WORK}/mod --log_dir={WORK}/log 2>&1 | grep -E "Epoch|RMSE|Done|rror" | tail -40

## 2. Held-out test (the same 300 configurations as for SNAP and POD)

In [ ]:
%cd /content/data
!ls {WORK}/mod
!python mace_test_eval.py {WORK}/mod/ti_mace_v3_stagetwo.model cuda | tee {WORK}/test_errors.txt

## 3. Melting point (solid–liquid coexistence, about 15–25 min per temperature)
At each temperature the two-phase box either crystallises (energy falls, T < Tm) or melts (energy rises, T > Tm). Start with the five temperatures below; afterwards add points between the last "solid grows" and the first "solid melts" to narrow the bracket. Each result is copied to Drive when it finishes.

In [ ]:
%cd /content/data
for T in (1700, 1800, 1900, 2000, 2100):
    if os.path.exists(f'{WORK}/melt_T{T}.dat'):
        print(f'T = {T} K already done'); continue
    !python mace_melt.py {WORK}/mod/ti_mace_v3_stagetwo.model {T} 30 cuda | tee -a {WORK}/melting.txt
    !cp melt_T{T}.dat {WORK}/

In [ ]:
!cat {WORK}/test_errors.txt {WORK}/melting.txt; ls {WORK}